# Lab 11: LoRA Fine-tuning

**Module 11** | Read `notes/11-lora-peft.pdf` before running this notebook.

Attach LoRA adapters to a small GPT-2 model and fine-tune on local instruction JSONL with a fraction of trainable weights.

## How to use this notebook

1. **Run cells top to bottom** (Shift+Enter). Do not skip markdown cells; they explain the code.
2. **Read before you run.** Each section tells you what the next code block does, which terms mean, and what the printed output should look like.
3. **Use the comments in code.** Lines starting with `#` explain what each step does in plain language.
4. **Check the output.** After each code cell, compare your printed numbers to the "What to look for" notes.

Code is complete. You are not expected to fill in missing sections or debug skeleton code.


## Runtime device (CPU or GPU)

**What this section does:** PyTorch can run math on your **CPU** (the main processor) or on an **NVIDIA GPU** through **CUDA**. GPUs are faster for neural networks because they run many small matrix operations in parallel.

**Key terms:**
- **CPU:** Always available. Training works but can be slower on large models.
- **GPU / CUDA:** Optional hardware acceleration. If you have an NVIDIA GPU and drivers installed, PyTorch will use it automatically.
- **VRAM:** GPU memory. If you run out, reduce batch size.

The next cell prints which device is active so you know what to expect for training speed.


In [ ]:
import torch

# Pick GPU if CUDA is available; otherwise fall back to CPU (labs still run).
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("Using CPU (no CUDA GPU detected).")
    print("All labs still run; training may take longer than on a GPU.")


## LoRA fine-tuning on GPT-2

**LoRA** (Low-Rank Adaptation) is a popular way to fine-tune large language models without updating every weight. Instead of retraining all 124 million parameters in GPT-2, you freeze the base model and attach small **low-rank adapters** to selected layers. Only those adapter matrices learn during training.

In this lab you will:
1. Load 500 instruction/response pairs from a local JSONL file.
2. Run a baseline completion with the unmodified GPT-2 model.
3. Attach LoRA adapters (`r=8`) to the attention layer.
4. Train for two short epochs and compare trainable vs total parameters.
5. Generate text again on the same prompt and compare before vs after.


## Key terms for this lab

| Term | Meaning |
|------|---------|
| **Fine-tuning** | Updating a pre-trained model on a new dataset so it learns task-specific behavior. |
| **LoRA** | Low-Rank Adaptation. A method that adds small trainable matrices to frozen layers instead of updating the full weight matrix. |
| **Low-rank adapters** | Small matrices (rank `r`) inserted next to a frozen weight. They approximate the change a full fine-tune would make, using far fewer parameters. |
| **Rank (r)** | The inner dimension of the low-rank decomposition. Smaller `r` means fewer trainable parameters but less capacity to adapt. |
| **PEFT** | Parameter-Efficient Fine-Tuning. A family of methods (including LoRA) that train only a tiny fraction of model weights. |
| **Causal LM** | A language model trained to predict the next token given all previous tokens (left-to-right). GPT-2 is a causal LM. |

Refer back to this table whenever a term appears in code or output.


### Step 1: Load and inspect instruction data

**What this section does:** Reads up to 500 rows from `datasets/instruction_sample.jsonl`. Each row has an `instruction`, optional `input`, and an `output` (the desired response).

**Why we do it:** Before tokenization or training, you should verify that field names and content look correct. A typo in the JSONL path or column names would break the rest of the lab silently.


**What to look for in the output**

- `Loaded 500 instruction examples` (or fewer if the file is shorter).
- Fields listed as `instruction`, `input`, `output`.
- Three sample records printed with readable instruction text and truncated outputs.


In [ ]:
import json
from pathlib import Path

import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

# Project root is one level above the labs/ folder where this notebook lives.
ROOT = Path("..").resolve()
DATA_PATH = ROOT / "datasets" / "instruction_sample.jsonl"

# Load at most 500 examples so training stays short on CPU or GPU.
records: list[dict] = []
with DATA_PATH.open(encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 500:
            break
        records.append(json.loads(line))

print(f"Loaded {len(records)} instruction examples from {DATA_PATH.name}")
print(f"Fields per record: {sorted(records[0].keys())}")
print()
print("Sample records (formatted):")
print("=" * 60)
for idx, rec in enumerate(records[:3]):
    print(f"--- Record {idx + 1} ---")
    print(f"instruction: {rec['instruction']}")
    if rec.get("input"):
        print(f"input:       {rec['input']}")
    # Truncate long outputs so the notebook stays readable.
    print(f"output:      {rec['output'][:120]}{'...' if len(rec['output']) > 120 else ''}")
    print()


### Step 2: Baseline generation (base model, no LoRA)

**What this section does:** Loads GPT-2 and runs greedy decoding on a fixed test prompt. Greedy decoding always picks the highest-probability next token.

**Why we do it:** You need a "before" snapshot. After LoRA training you will run the exact same prompt so you can see whether the adapter changed the model's behavior.


**What to look for in the output**

- Model loads without error.
- The printed completion is coherent English but may not follow the instruction format well (base GPT-2 was not trained for chat-style prompts).
- Save this output mentally; you will compare it to the fine-tuned completion later.


In [ ]:
MODEL_NAME = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# GPT-2 has no dedicated pad token; reuse the end-of-sequence token for padding.
tokenizer.pad_token = tokenizer.eos_token

TEST_PROMPT = "### Instruction:\nExplain reinforcement learning\n### Response:\n"

# Load the full base model (all weights trainable in principle, but we will freeze them soon).
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
base_model.eval().to(device)

inputs = tokenizer(TEST_PROMPT, return_tensors="pt").to(device)
with torch.no_grad():
    base_ids = base_model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=False,  # Greedy: always pick argmax token.
        pad_token_id=tokenizer.eos_token_id,
    )

print("Prompt:")
print(TEST_PROMPT)
print()
print("Base model completion (before LoRA fine-tuning):")
print("-" * 60)
print(tokenizer.decode(base_ids[0], skip_special_tokens=True))
print("-" * 60)


### Step 3: Attach LoRA adapters

**What this section does:** Wraps the frozen GPT-2 model with PEFT LoRA adapters. `r=8` sets the rank of the low-rank matrices. `target_modules=["c_attn"]` attaches adapters to the fused query-key-value projection in GPT-2's attention blocks.

**Why we do it:** Full fine-tuning of GPT-2 requires storing gradients for every parameter, which uses a lot of GPU memory. LoRA trains only the adapter weights (often well under 1% of total parameters) while the base weights stay fixed. The idea: a weight update `delta_W` can often be approximated by a product of two small matrices `A @ B` where `A` and `B` have only `r` columns/rows.


**What to look for in the output**

- Trainable parameter count is a tiny fraction of total (typically well below 1%).
- `print_trainable_parameters()` from PEFT shows the same ratio.
- Total parameters should still be around 124M (GPT-2 small).


In [ ]:
from peft import LoraConfig, get_peft_model

# LoRA hyperparameters:
#   r=8          : rank of the low-rank adapter matrices
#   lora_alpha=16: scaling factor (often set to 2*r)
#   target_modules: which layer names receive adapters
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["c_attn"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# get_peft_model freezes base weights and marks only adapter params as trainable.
model = get_peft_model(base_model, lora_config)
model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
pct = 100.0 * trainable / total

print("Parameter summary:")
print(f"  Trainable: {trainable:,}")
print(f"  Total:     {total:,}")
print(f"  Trainable fraction: {pct:.4f}%")
print()
model.print_trainable_parameters()


### Step 4: Short training loop

**What this section does:** Formats each record as a single text string, tokenizes it, and trains the LoRA adapters for two epochs with AdamW. Loss is standard next-token cross-entropy; padding positions are masked with label `-100`.

**Why we do it:** Two epochs over 500 examples is enough to demonstrate the workflow and often produces a visible loss drop. In production you would train longer, use a validation split, and track perplexity or task metrics.


**What to look for in the output**

- `Training examples: 500` and `Batches per epoch: 125` (with batch size 4).
- Average loss printed for epoch 1 and epoch 2; epoch 2 should usually be lower.
- `Loss decrease` line at the end should be positive (loss went down).


In [ ]:
def format_example(rec: dict) -> str:
    # Build a simple instruction-tuning template the model can learn to continue.
    prompt = rec["instruction"]
    if rec.get("input"):
        prompt += "\n" + rec["input"]
    return (
        f"### Instruction:\n{prompt}\n### Response:\n{rec['output']}"
        f"{tokenizer.eos_token}"
    )


class InstructionDataset(Dataset):
    def __init__(self, rows: list[dict], max_length: int = 128):
        self.texts = [format_example(r) for r in rows]
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int) -> dict:
        enc = tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt",
        )
        input_ids = enc["input_ids"].squeeze(0)
        attention_mask = enc["attention_mask"].squeeze(0)
        # Labels equal input_ids for causal LM; mask padding so loss ignores padded tokens.
        labels = input_ids.clone()
        labels[attention_mask == 0] = -100
        return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}


def collate(batch: list[dict]) -> dict:
    return {
        "input_ids": torch.stack([b["input_ids"] for b in batch]),
        "attention_mask": torch.stack([b["attention_mask"] for b in batch]),
        "labels": torch.stack([b["labels"] for b in batch]),
    }


dataset = InstructionDataset(records)
loader = DataLoader(dataset, batch_size=4, shuffle=True, collate_fn=collate)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)

print(f"Training examples: {len(dataset)}")
print(f"Batches per epoch: {len(loader)}")
print(f"Batch size: 4 | Learning rate: 2e-4 | Epochs: 2")
print()

model.train()
epoch_losses: list[float] = []
for epoch in range(2):
    epoch_loss = 0.0
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        out = model(**batch)
        loss = out.loss
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    avg = epoch_loss / len(loader)
    epoch_losses.append(avg)
    print(f"Epoch {epoch + 1}/2, average loss: {avg:.4f}")

print()
print("Epoch loss summary:")
for i, loss_val in enumerate(epoch_losses, start=1):
    print(f"  Epoch {i}: {loss_val:.4f}")
if len(epoch_losses) >= 2:
    delta = epoch_losses[0] - epoch_losses[-1]
    print(f"  Loss decrease (epoch 1 to {len(epoch_losses)}): {delta:.4f}")


### Step 5: Fine-tuned generation on the same prompt

**What this section does:** Switches the model to eval mode and runs greedy decoding on `TEST_PROMPT`, the same string used before LoRA training.

**Why we do it:** Side-by-side comparison is the quickest sanity check. Even a short fine-tune can shift tone, structure, or topical focus on instruction-style prefixes.


**What to look for in the output**

- The prompt block matches the baseline section exactly.
- The fine-tuned completion may differ in wording, length, or relevance to "reinforcement learning."
- Compare this output to the base model completion printed in Step 2.


In [ ]:
model.eval()
inputs = tokenizer(TEST_PROMPT, return_tensors="pt").to(device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

print("Prompt:")
print(TEST_PROMPT)
print()
print("Fine-tuned model completion (after LoRA):")
print("-" * 60)
print(tokenizer.decode(output_ids[0], skip_special_tokens=True))
print("-" * 60)


### Step 6: Evaluation summary

**What this section does:** Prints a short checklist of training statistics and reminds you what to compare.

**Why we do it:** Numbers alone do not tell the whole story. Decreasing loss plus a visibly different completion on the same prompt suggests the adapters learned something useful.


**What to look for in the output**

- Examples trained on: 500.
- Trainable parameters still show the small fraction from Step 3.
- Final epoch loss matches the last epoch from Step 4.
- Printed checklist references both completions above.


In [ ]:
print("LoRA fine-tuning evaluation:")
print(f"  Examples trained on: {len(records)}")
print(f"  Trainable parameters: {trainable:,} ({pct:.4f}% of {total:,})")
print(f"  Final epoch loss: {epoch_losses[-1]:.4f}")
print()
print("Compare the two completions above:")
print("  1. Base model (before LoRA)")
print("  2. Fine-tuned model (after 2 epochs)")
print()
print("A successful run shows decreasing loss and a visibly different fine-tuned response.")
